In [ ]:
!pip install biopython networkx numpy pandas matplotlib -q

import os
import urllib.request
import numpy as np
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

print("Done!")


In [ ]:
# proteins we're gonna look at
protein_ids = ['1ubq', '1hrc', '1csp', '1fkb', '2abd']
pdb_dir = 'pdb_files'
os.makedirs(pdb_dir, exist_ok=True)

def grab_pdb(pdb_id):
    """download a pdb file if we don't have it yet"""
    pdb_id = pdb_id.lower()
    filepath = os.path.join(pdb_dir, f"{pdb_id}.pdb")
    if os.path.exists(filepath):
        print(f"  [CACHED] {pdb_id}.pdb")
        return filepath
    url = f"https://files.rcsb.org/download/{pdb_id.upper()}.pdb"
    urllib.request.urlretrieve(url, filepath)
    print(f"  [OK] {pdb_id}.pdb downloaded")
    return filepath

pdb_paths = {pid: grab_pdb(pid) for pid in protein_ids}
print("All downloads done!")


In [ ]:
def extract_ca_coordinates(pdb_filepath):
    """pull out C-alpha atoms from a pdb file"""
    ca_atoms = []
    seen_residues = set()
    target_chain = None

    with open(pdb_filepath, 'r') as f:
        for line in f:
            record = line[:6].strip()
            if record == 'ENDMDL':
                break
            if record != 'ATOM':
                continue

            atom_name = line[12:16].strip()
            alt_loc   = line[16].strip()
            chain_id  = line[21].strip()
            res_name  = line[17:20].strip()

            # skip weird stuff
            if alt_loc not in ('', 'A'):
                continue
            if atom_name != 'CA':
                continue

            try:
                res_seq = int(line[22:26].strip())
                x = float(line[30:38])
                y = float(line[38:46])
                z = float(line[46:54])
            except ValueError:
                continue

            # stick to one chain
            if target_chain is None:
                target_chain = chain_id
            if chain_id != target_chain:
                continue
            if res_seq in seen_residues:
                continue

            seen_residues.add(res_seq)
            ca_atoms.append((res_seq, res_name, x, y, z))

    ca_atoms.sort(key=lambda t: t[0])
    return ca_atoms

protein_coords = {}
for pid in protein_ids:
    coords = extract_ca_coordinates(pdb_paths[pid])
    protein_coords[pid] = coords
    print(f"{pid.upper()}: {len(coords)} Cα residues")


In [ ]:
# cutoff distance for the RIG model (in angstroms)
rig_cutoff = 8.0

def build_rig(ca_atoms, cutoff=rig_cutoff):
    """build a residue interaction graph from CA atoms"""
    n = len(ca_atoms)
    G = nx.Graph()
    for idx, (res_seq, res_name, x, y, z) in enumerate(ca_atoms):
        G.add_node(idx, res_seq=res_seq, res_name=res_name)
    for i in range(n):
        xi, yi, zi = ca_atoms[i][2], ca_atoms[i][3], ca_atoms[i][4]
        for j in range(i+1, n):
            xj, yj, zj = ca_atoms[j][2], ca_atoms[j][3], ca_atoms[j][4]
            dist = np.sqrt((xi-xj)**2 + (yi-yj)**2 + (zi-zj)**2)
            if dist <= cutoff:
                G.add_edge(i, j, distance=dist)
    return G

rig_graphs = {}
for pid in protein_ids:
    G = build_rig(protein_coords[pid])
    rig_graphs[pid] = G
    print(f"{pid.upper()} RIG → Nodes: {G.number_of_nodes()}, Edges: {G.number_of_edges()}, Avg Deg: {np.mean([d for _,d in G.degree()]):.2f}")


In [ ]:
# sequence distance threshold for LIN model
lin_seq_threshold = 12
lin_dist_cutoff   = 8.0

def build_lin(ca_atoms, seq_threshold=lin_seq_threshold, dist_cutoff=lin_dist_cutoff):
    """build a local interaction network from CA atoms"""
    n = len(ca_atoms)
    G = nx.Graph()
    for idx, (res_seq, res_name, x, y, z) in enumerate(ca_atoms):
        G.add_node(idx, res_seq=res_seq, res_name=res_name)
    for i in range(n):
        xi, yi, zi = ca_atoms[i][2], ca_atoms[i][3], ca_atoms[i][4]
        for j in range(i + seq_threshold, n):
            xj, yj, zj = ca_atoms[j][2], ca_atoms[j][3], ca_atoms[j][4]
            dist = np.sqrt((xi-xj)**2 + (yi-yj)**2 + (zi-zj)**2)
            if dist <= dist_cutoff:
                G.add_edge(i, j, distance=dist)
    return G

lin_graphs = {}
for pid in protein_ids:
    G = build_lin(protein_coords[pid])
    lin_graphs[pid] = G
    print(f"{pid.upper()} LIN → Nodes: {G.number_of_nodes()}, Edges: {G.number_of_edges()}, Avg Deg: {np.mean([d for _,d in G.degree()]):.2f}")


In [ ]:
def calc_network_props(G, label=''):
    """compute basic network properties"""
    n_nodes = G.number_of_nodes()
    n_edges = G.number_of_edges()
    avg_deg = np.mean([d for _, d in G.degree()])
    C = nx.average_clustering(G)

    if nx.is_connected(G):
        L = nx.average_shortest_path_length(G)
    else:
        # grab the biggest connected piece
        lcc = max(nx.connected_components(G), key=len)
        subG = G.subgraph(lcc).copy()
        L = nx.average_shortest_path_length(subG)
        print(f"  [{label}] Not connected, using LCC ({len(lcc)}/{n_nodes} nodes)")

    return {'Nodes': n_nodes, 'Edges': n_edges,
            'Avg Degree': round(avg_deg,3), 'L': round(L,4), 'C': round(C,4)}

results_rig = {}
results_lin = {}

for pid in protein_ids:
    print(f"\nProcessing {pid.upper()}...")
    results_rig[pid] = calc_network_props(rig_graphs[pid], f'{pid}-RIG')
    results_lin[pid] = calc_network_props(lin_graphs[pid], f'{pid}-LIN')

print("\nDone!")


In [ ]:
rows = []
for pid in protein_ids:
    for model, res in [('RIG', results_rig[pid]), ('LIN', results_lin[pid])]:
        rows.append({'PDB ID': pid.upper(), 'Model': model,
                     'Nodes': res['Nodes'], 'Edges': res['Edges'],
                     'Avg Degree': res['Avg Degree'],
                     'Path Length (L)': res['L'],
                     'Clustering (C)': res['C']})

df = pd.DataFrame(rows).set_index(['PDB ID','Model'])
print(df.to_string())


In [ ]:
pids_upper = [p.upper() for p in protein_ids]
L_rig = [results_rig[p]['L'] for p in protein_ids]
L_lin = [results_lin[p]['L'] for p in protein_ids]
C_rig = [results_rig[p]['C'] for p in protein_ids]
C_lin = [results_lin[p]['C'] for p in protein_ids]

x = np.arange(len(protein_ids))
width = 0.35

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle("RIG vs LIN — Network Properties", fontsize=14, fontweight='bold')

for ax, Y_rig, Y_lin, ylabel, title in zip(
    axes,
    [L_rig, C_rig], [L_lin, C_lin],
    ['Characteristic Path Length (L)', 'Clustering Coefficient (C)'],
    ['Path Length L', 'Clustering Coefficient C']
):
    b1 = ax.bar(x - width/2, Y_rig, width, label='RIG', color='steelblue', edgecolor='k', alpha=0.85)
    b2 = ax.bar(x + width/2, Y_lin, width, label='LIN', color='coral',     edgecolor='k', alpha=0.85)
    ax.set_xticks(x); ax.set_xticklabels(pids_upper)
    ax.set_xlabel('Protein (PDB ID)', fontsize=11)
    ax.set_ylabel(ylabel, fontsize=11)
    ax.set_title(title, fontsize=12)
    ax.legend(); ax.grid(axis='y', linestyle='--', alpha=0.5)

plt.tight_layout()
plt.savefig('Q3_RIG_vs_LIN.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
fig, axes = plt.subplots(1, 5, figsize=(22, 4))
fig.suptitle("Degree Distribution: RIG vs LIN", fontsize=13, fontweight='bold')

for idx, pid in enumerate(protein_ids):
    ax = axes[idx]
    deg_rig = [d for _, d in rig_graphs[pid].degree()]
    deg_lin = [d for _, d in lin_graphs[pid].degree()]
    ax.hist(deg_rig, bins=20, alpha=0.6, label='RIG', color='steelblue', density=True, edgecolor='k')
    ax.hist(deg_lin, bins=20, alpha=0.6, label='LIN', color='coral',     density=True, edgecolor='k')
    ax.set_title(pid.upper()); ax.set_xlabel('Degree k'); ax.set_ylabel('P(k)')
    ax.legend(); ax.grid(linestyle='--', alpha=0.4)

plt.tight_layout()
plt.savefig('Q3_degree_dist.png', dpi=150, bbox_inches='tight')
plt.show()
